# UMAP Pipeline — Zeng / Zhuang / ISD
Loads embeddings from model directory, builds combined adata, plots UMAP.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import os

## Configuration — edit these paths

In [ ]:
# Path to the model embeddings directory (contains subfolders: zeng/, zhuang/, isd/)
MODEL_DIR = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/concept_embeddings/models/small_weighted__hgld4ax1_last"

# Which embedding to use for UMAP
EMBEDDING_KEY = "concept_cls_embedding"  # or concept_mean_embedding

# Cells per dataset after subsampling (set None to disable)
SUBSAMPLE = 50000

# Output dir for UMAP pngs
OUTPUT_DIR = "/p/scratch/cjinm16/dipippo1/scConcept/umaps"

In [ ]:
DATASET_CONFIGS = {
    "zeng": {
        "adata_path": "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zeng.h5ad",
        "annotation_path": "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zeng/results/merfish/cell_type_annotation/adata_obs_annotated.csv",
        "cell_type_col": "cell_type_mmc_raw",
        "cell_id_suffix": "-Zeng",
    },
    "zhuang": {
        "adata_path": "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1.h5ad",
        "annotation_path": "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zhuang/results/merfish/cell_type_annotation/adata_obs_annotated.csv",
        "cell_type_col": "cell_type_mmc_raw",
        "cell_id_suffix": "-Zhuang-ABCA-1",
    },
    "isd": {
        "adata_path": "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/concept_embeddings/hvg_isd_normed.h5ad",
        "annotation_path": None,
        "cell_type_col": "cell_type_mmc_raw_revised",
        "cell_id_suffix": None,
    },
}

CELL_TYPE_PALETTE = {
    'ABCs': '#023FA5', 'Astrocytes': '#7D87B9', 'Astroependymal': '#BEC1D4',
    'BAMs': '#D6BCC0', 'Bergmann': '#BB7784', 'Choroid-Plexus': '#8E063B',
    'ECs': '#4A6FE3', 'Ependymal': '#8595E1', 'Immune-Other': '#B5BBE3',
    'Microglia': '#E6AFB9', 'Neurons-Dopa': '#E07B91', 'Neurons-Gaba': '#D33F6A',
    'Neurons-Glut': '#11C638', 'Neurons-Glyc-Gaba': '#8DD593',
    'Neurons-Granule-Immature': '#C6DEC7', 'Neurons-Other': '#EAD3C6',
    'OECs': '#F0B98D', 'OPCs': '#EF9708', 'Oligodendrocytes': '#0FCFC0',
    'Pericytes': '#9CDED6', 'SMCs': '#D5EAE7', 'Tanycytes': '#F3E1EB',
    'Undefined': '#F6C4E1', 'Unknown': '#F6C4E1', 'VLMCs': '#F79CD4'
}

## Step 1 — Load embeddings from .npy files (like evaluate.py)

In [ ]:
def add_concept_embeddings(adata, embedding_path, name="Dataset"):
    """Load .npy embeddings and align to adata by obs_names intersection."""
    cell_ids = np.load(f"{embedding_path}/cell_ids.npy", allow_pickle=True).astype(str)
    emb_mean = np.load(f"{embedding_path}/cell_embs_mean.npy")
    emb_cls  = np.load(f"{embedding_path}/cell_embs_cls.npy")

    df_mean = pd.DataFrame(emb_mean, index=cell_ids)
    df_cls  = pd.DataFrame(emb_cls,  index=cell_ids)

    common_cells = adata.obs_names.intersection(cell_ids)
    print(f"{name}: {len(common_cells):,} / {adata.n_obs:,} cells have embeddings")

    adata = adata[common_cells].copy()
    adata.obsm["concept_mean_embedding"] = df_mean.loc[adata.obs_names].to_numpy()
    adata.obsm["concept_cls_embedding"]  = df_cls.loc[adata.obs_names].to_numpy()
    return adata

## Step 2 — Load datasets + annotations

In [ ]:
def load_dataset(name, model_dir, subsample=None):
    cfg = DATASET_CONFIGS[name]
    embedding_path = os.path.join(model_dir, name)
    print(f"\n--- Loading {name} ---")

    adata = sc.read_h5ad(cfg["adata_path"])
    adata.X = None  # drop gene matrix to save memory

    adata = add_concept_embeddings(adata, embedding_path, name=name)

    # Attach cell_type
    if cfg["annotation_path"] is not None:
        ann = pd.read_csv(cfg["annotation_path"])
        suffix = cfg.get("cell_id_suffix")

        # Strip suffix from obs_names to match annotation cell_ids
        if suffix:
            adata.obs["cell_id_stripped"] = adata.obs_names.str.replace(suffix, "", regex=False)
        else:
            adata.obs["cell_id_stripped"] = adata.obs_names

        ann = ann[["cell_id", cfg["cell_type_col"]]].rename(
            columns={cfg["cell_type_col"]: "cell_type", "cell_id": "cell_id_stripped"}
        )
        adata.obs = adata.obs.merge(ann, on="cell_id_stripped", how="left")
        adata.obs.index = adata.obs_names
    else:
        col = cfg["cell_type_col"]
        if col in adata.obs.columns:
            adata.obs["cell_type"] = adata.obs[col]
        elif "cell_type" not in adata.obs.columns:
            adata.obs["cell_type"] = "Unknown"

    adata.obs["cell_type"] = adata.obs["cell_type"].fillna("Unknown").astype(str).astype("category")
    adata.obs["dataset"] = name.upper()

    if subsample and adata.n_obs > subsample:
        print(f"   Subsampling {name}: {adata.n_obs:,} -> {subsample:,}")
        sc.pp.subsample(adata, n_obs=subsample, random_state=0)

    print(f"   Final: {adata.n_obs:,} cells")
    return adata

In [ ]:
zeng    = load_dataset("zeng",   MODEL_DIR, subsample=SUBSAMPLE)
zhuang  = load_dataset("zhuang", MODEL_DIR, subsample=SUBSAMPLE)
isd     = load_dataset("isd",    MODEL_DIR, subsample=SUBSAMPLE)

## Step 3 — Concatenate

In [ ]:
adata_ref = ad.concat(
    {"ZENG": zeng, "ZHUANG": zhuang, "ISD": isd},
    join="outer",
    label="dataset",
    index_unique="-",
)
adata_ref.obs_names_make_unique()

print(f"Combined: {adata_ref.n_obs:,} cells")
print(adata_ref.obs["dataset"].value_counts())

## Step 4 — Neighbors + UMAP

In [ ]:
sc.pp.neighbors(adata_ref, n_neighbors=30, use_rep=EMBEDDING_KEY)
sc.tl.umap(adata_ref, min_dist=0.3, random_state=0)

## Step 5 — Plot

In [ ]:
# Plot by dataset
fig = sc.pl.umap(
    adata_ref,
    color="dataset",
    title=f"UMAP by Dataset ({EMBEDDING_KEY})",
    frameon=False,
    show=False,
    return_fig=True,
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, "umap_dataset.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Plot by cell type
cats = adata_ref.obs["cell_type"].unique()
use_palette = {k: v for k, v in CELL_TYPE_PALETTE.items() if k in cats}

fig = sc.pl.umap(
    adata_ref,
    color="cell_type",
    palette=use_palette,
    title=f"UMAP by Cell Type ({EMBEDDING_KEY})",
    frameon=False,
    show=False,
    return_fig=True,
)
fig.savefig(os.path.join(OUTPUT_DIR, "umap_celltype.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)